# StockLens - Stock Price Analysis

**Data Source:** Yahoo Finance (live, via `yfinance`)  
**Stocks:** Apple (AAPL), Microsoft (MSFT), Tesla (TSLA), Amazon (AMZN), Google (GOOGL)  
**Period:** 2020 - Present  
**Goal:** Analyse closing price trends, moving averages, daily returns, volatility and correlations.

---

## 1. Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from datetime import date

matplotlib.rcParams['font.family']        = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

COLORS  = ['#3b82f6', '#f59e0b', '#10b981', '#ef4444', '#8b5cf6']
TICKERS = ['AAPL', 'MSFT', 'TSLA', 'AMZN', 'GOOGL']
OUT     = 'outputs'

print('Libraries loaded!')

## 2. Download Live Stock Data

In [ ]:
START = '2020-01-01'
END   = str(date.today())

raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)
df  = raw['Close'].copy()
df.dropna(inplace=True)

end_year = df.index[-1].year
returns  = df.pct_change().dropna()

print(f'Period : {df.index[0].date()} to {df.index[-1].date()}')
print(f'Shape  : {df.shape}')
df.head()

## 3. Closing Price Trends

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for ticker, color in zip(TICKERS, COLORS):
    ax.plot(df.index, df[ticker], label=ticker, color=color, linewidth=1.8)
ax.set_title(f'Closing Price Trends (2020 - {end_year})', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f'{OUT}/closing_prices.png', dpi=150)
plt.show()

## 4. Normalised Performance (Base = 100)

In [ ]:
norm = (df / df.iloc[0]) * 100
fig, ax = plt.subplots(figsize=(14, 6))
for ticker, color in zip(TICKERS, COLORS):
    ax.plot(norm.index, norm[ticker], label=ticker, color=color, linewidth=1.8)
ax.axhline(100, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_title('Normalised Performance (Base = 100, Jan 2020)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Indexed Value')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f'{OUT}/normalised_performance.png', dpi=150)
plt.show()

## 5. Moving Averages - Apple (AAPL)

In [ ]:
aapl = df[['AAPL']].copy()
aapl['MA30'] = aapl['AAPL'].rolling(30).mean()
aapl['MA90'] = aapl['AAPL'].rolling(90).mean()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(aapl.index, aapl['AAPL'],  label='AAPL Close', color='#3b82f6', linewidth=1.5, alpha=0.8)
ax.plot(aapl.index, aapl['MA30'],  label='30-Day MA',  color='#f59e0b', linewidth=2)
ax.plot(aapl.index, aapl['MA90'],  label='90-Day MA',  color='#ef4444', linewidth=2)
ax.set_title('Apple (AAPL) - Price and Moving Averages', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT}/aapl_moving_averages.png', dpi=150)
plt.show()

## 6. Daily Return Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, (ticker, color) in enumerate(zip(TICKERS, COLORS)):
    ax = axes[i]
    ax.hist(returns[ticker], bins=80, color=color, edgecolor='none', alpha=0.85)
    ax.axvline(0, color='white', linestyle='--', linewidth=0.8)
    ax.set_title(ticker, fontsize=13, fontweight='bold')
    ax.set_xlabel('Daily Return', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
axes[-1].set_visible(False)
fig.suptitle('Daily Return Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/daily_returns.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 30-Day Rolling Volatility

In [ ]:
vol = returns.rolling(30).std() * np.sqrt(252)
fig, ax = plt.subplots(figsize=(14, 5))
for ticker, color in zip(TICKERS, COLORS):
    ax.plot(vol.index, vol[ticker], label=ticker, color=color, linewidth=1.5)
ax.set_title('30-Day Rolling Volatility (Annualised)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Volatility')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT}/volatility.png', dpi=150)
plt.show()

## 8. Correlation Heatmap

In [ ]:
corr = returns.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax, vmin=0, vmax=1, annot_kws={'size': 12})
ax.set_title('Return Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/correlation_heatmap.png', dpi=150)
plt.show()

## 9. Key Statistics

In [ ]:
summary = pd.DataFrame({
    'Start Price ($)' : df.iloc[0].round(2),
    'End Price ($)'   : df.iloc[-1].round(2),
    'Total Return (%)': (((df.iloc[-1] / df.iloc[0]) - 1) * 100).round(2),
    'Avg Daily Ret (%)': (returns.mean() * 100).round(3),
    'Daily Vol (%)'   : (returns.std() * 100).round(3),
    'Max Price ($)'   : df.max().round(2),
    'Min Price ($)'   : df.min().round(2),
})
print('=' * 55)
print('          STOCKLENS - KEY STATISTICS')
print('=' * 55)
summary

---
**Tools:** Python - Pandas - NumPy - Matplotlib - Seaborn - yfinance  
**Author:** Berke Arda Turk